# Tutorial: how SADIE works

This is a slow, step-by-step introduction to SADIE (`sadiepy`). It assumes you know some
Python and NumPy, but no wave theory. Every object is introduced on its own, with what
it represents physically, how to create it, what you can ask it, and what can go wrong.
For a quick visual overview instead, see the [tour](02_tour.ipynb).

## The big picture

SADIE answers one question: **if a wave of a given size enters a patch of vegetation,
how big is it when it comes out?**

A wave carries energy forward. Inside a patch of plants, the water moves back and forth
past every root, stem and leaf. Each of them pushes back on the water (drag), and that
drag turns wave energy into turbulence and heat. SADIE adds up this loss from the seabed
to the water surface and follows the wave as it shrinks along the patch.

To do that it needs four ingredients, and it gives back one result:

| Ingredient | SADIE object | What it describes |
|---|---|---|
| The water | `sd.Water` | density and viscosity (from temperature and salinity) |
| The wave | `sd.RegularWave` | wave height, period and water depth |
| The plants | `sd.Vegetation` made of `sd.Layer`s | how much plant there is, at each height above the bed |
| The drag law | a number, or a closure from `sd.drag` | how strongly each plant part resists the flow ($C_D$) |
| **The result** | `sd.PropagationResult` from `sd.propagate` | wave height along the patch, damping factor, energy budget, diagnostics |

The notebook follows that order:

1. [Setup and conventions](#0-setup-and-conventions)
2. [The water](#1-the-water)
3. [The wave](#2-the-wave)
4. [The vegetation](#3-the-vegetation)
5. [The drag coefficient](#4-the-drag-coefficient)
6. [Propagating a wave through the patch](#5-propagating-a-wave-through-the-patch)
7. [When the drag coefficient depends on the wave](#6-when-the-drag-coefficient-depends-on-the-wave)
8. [Scaling between the lab and the field](#7-scaling-between-the-lab-and-the-field)
9. [Errors and what they mean](#8-errors-and-what-they-mean)
10. [Putting it together: a small study](#9-putting-it-together-a-small-study)
11. [Cheat sheet](#10-cheat-sheet)

Equation numbers such as "Eq. 3.7" refer to the [theory](../theory.md) page, where every
formula is derived.

## 0. Setup and conventions

The convention is to import the package as `sd`. Everything you need for a basic
calculation is available at the top level: `sd.Water`, `sd.RegularWave`, `sd.Layer`,
`sd.Vegetation`, `sd.propagate`, `sd.drag` and `sd.presets`.

In [ ]:
import dataclasses
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

import sadiepy as sd

print("sadiepy version", sd.__version__)

# Plot style used throughout (colour-blind-safe colours).
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "0.92", "axes.axisbelow": True, "font.size": 10,
    "legend.frameon": False,
})
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"  # three series colours
PART = {"root": "#6250d6", "stem": "#eda100", "canopy": "#008300"}

Four conventions hold everywhere in SADIE. Keeping them in mind avoids most mistakes.

1. **SI units, always.** Metres, seconds, kilograms, °C for temperature and g/kg for
   salinity. There are no unit options; a period of 1.4 means 1.4 s.
2. **Heights are measured up from the bed.** The symbol is $s$: $s = 0$ is the bed,
   $s = h$ is the still-water surface ($h$ is the water depth). Many papers use $z$
   from $-h$ to $0$ instead; convert with $s = z + h$.
3. **Distance along the patch is $x$**, in metres from the patch's leading edge (where
   the wave enters). The wave travels in the $+x$ direction.
4. **$H$ is the wave height**, trough to crest. The amplitude is $a = H/2$.

Inputs are immutable ("frozen") dataclasses: once you create a wave or a layer, it cannot
change. To try a different value you make a new object, often with
`dataclasses.replace`. This keeps results reproducible: an object you passed to the
solver is guaranteed to be the one that produced the result.

SADIE never fails silently when a formula is used outside its range. It returns a value
**and** emits a `sd.ValidityWarning` (or its subclass `sd.ExtrapolationWarning` for
empirical fits). This notebook shows those warnings rather than hiding them.

## 1. The water

Water enters the physics in two places:

- its **density** $\rho$ (kg/m³) sets how much energy a wave of a given height carries
  and how hard the water pushes on a plant;
- its **kinematic viscosity** $\nu$ (m²/s) enters only the Reynolds number, which some
  drag laws use.

`sd.Water` computes both from temperature and salinity with the seawater correlations of
Sharqawy et al. (2010) (theory §1). With no arguments you get fresh water at 20 °C, which
is what most flumes use.

In [ ]:
fresh = sd.Water()  # defaults: temperature=20.0 °C, salinity=0.0 g/kg
sea = sd.Water(temperature=15.0, salinity=35.0)  # typical open-ocean water

for label, w in [("fresh, 20 °C", fresh), ("sea, 15 °C", sea)]:
    print(f"{label:13s} rho = {w.rho:8.2f} kg/m³   mu = {w.mu:.3e} Pa s   nu = {w.nu:.3e} m²/s")

`rho`, `mu` (dynamic viscosity) and `nu` (kinematic viscosity, $\nu = \mu/\rho$) are
computed properties: they are evaluated when you ask for them.

Because `Water` is frozen, you cannot change its temperature in place. Use
`dataclasses.replace` to get a modified copy:

In [ ]:
try:
    fresh.temperature = 10.0
except dataclasses.FrozenInstanceError as err:
    print("Cannot modify:", err)

cold = dataclasses.replace(fresh, temperature=10.0)
print(f"cold fresh water: rho = {cold.rho:.2f} kg/m³, nu = {cold.nu:.3e} m²/s")

Outside the range where the correlations were fitted, `Water` still returns a value but
warns. Here is a temperature above 180 °C, caught and printed:

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    hot = sd.Water(temperature=200.0)
for w in caught:
    print(f"{w.category.__name__}: {w.message}")

**When does the choice of water matter?** Density changes by only about 2–3% between
fresh and sea water, so it hardly changes the damping. Viscosity matters more (it roughly
doubles between 30 °C and 0 °C), but only if your drag law depends on the Reynolds
number. If you don't pass a `Water` to the solver, it uses `sd.Water()`.

## 2. The wave

### 2.1 Creating a wave

A `RegularWave` is a single, repeating wave (all crests the same height) in water of
constant depth. It needs three numbers:

- `H`: wave height in m (trough to crest),
- `T`: wave period in s (time between two crests passing a fixed point),
- `depth`: still-water depth $h$ in m.

We'll use a wave from a typical laboratory flume: 8 cm high, 1.4 s period, in 40 cm of
water.

In [ ]:
wave = sd.RegularWave(H=0.08, T=1.4, depth=0.4)
wave

### 2.2 Wavelength: the dispersion relation

The height and period are what you set on a wave maker, but the physics also needs the
**wavelength** $L$ (crest-to-crest distance) or, equivalently, the **wavenumber**
$k = 2\pi/L$. These are not independent: in linear wave theory, period, depth and
wavelength are tied together by the dispersion relation (theory Eq. 2.2):

$$\omega^2 = g\,k\,\tanh(kh), \qquad \omega = 2\pi/T.$$

There is no closed-form solution for $k$, so SADIE solves it numerically (Newton's
method, accurate to about $10^{-12}$) as soon as you create the wave. The result is
stored as `wave.k`.

In [ ]:
print(f"angular frequency omega = {wave.omega:.4f} rad/s")
print(f"wavenumber        k     = {wave.k:.4f} rad/m")
print(f"wavelength        L     = {wave.L:.4f} m")
print(f"relative depth    kh    = {wave.kh:.4f}")
print(f"amplitude         a     = {wave.a:.4f} m")

# Check: the dispersion relation holds to machine precision.
residual = wave.omega**2 - sd.GRAVITY * wave.k * math.tanh(wave.kh)
print(f"\ndispersion residual: {residual:.1e} (rad/s)²")

The number $kh$ (relative depth) tells you what kind of wave you have:

- $kh < \pi/10 \approx 0.3$: **shallow-water** wave. The water moves almost equally at
  all depths.
- $kh > \pi \approx 3.1$: **deep-water** wave. The motion is concentrated near the
  surface and does not feel the bed.
- In between: **intermediate depth**, where most flume and coastal waves sit.

Our wave has $kh \approx 1$, so it is intermediate. The figure shows how the wavelength
grows with the period, and how the depth limits it.

In [ ]:
periods = np.linspace(0.6, 3.0, 100)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
for depth, color in [(0.3, C1), (0.5, C2), (1.0, C3)]:
    lengths = [sd.RegularWave(0.01, T, depth).L for T in periods]
    ax.plot(periods, lengths, color=color, lw=2, label=f"h = {depth} m")
ax.plot(periods, sd.GRAVITY * periods**2 / (2 * np.pi), color="0.5", ls="--",
        label="deep-water limit $gT^2/2\\pi$")
ax.set_xlabel("period T (s)")
ax.set_ylabel("wavelength L (m)")
ax.set_title("Longer periods give longer waves; shallow water shortens them")
ax.legend()
plt.show()

### 2.3 How fast the wave travels

Two speeds matter:

- the **phase speed** $c = L/T$ is how fast a crest moves;
- the **group speed** $c_g = n\,c$ is how fast the wave's *energy* moves (theory Eq. 2.7).
  The factor $n$ is between ½ (deep water) and 1 (shallow water).

Damping is about energy, so the group speed is the one that appears in the solver.

In [ ]:
print(f"phase speed c   = {wave.c:.4f} m/s")
print(f"factor      n   = {wave.n:.4f}")
print(f"group speed c_g = {wave.cg:.4f} m/s")

### 2.4 How the water moves under the wave

As a wave passes, each bit of water moves back and forth horizontally (and up and down).
The plants feel the **horizontal velocity**. Its amplitude at height $s$ above the bed is
(theory Eq. 2.8)

$$u_m(s) = \frac{H}{2}\,\omega\,\frac{\cosh(ks)}{\sinh(kh)}.$$

It is largest at the surface and smallest at the bed. `wave.u_m(s)` evaluates it at any
height (or an array of heights). Two related quantities:

- `wave.u_bar`: the **depth-averaged** velocity amplitude $\bar u = \omega a/(kh)$
  (theory Eq. 2.9). Many drag laws use this as their velocity scale.
- `wave.orbital_excursion(s)`: how far the water moves to each side, $A_w = u_m/\omega$
  (theory Eq. 2.10).

In [ ]:
s = np.linspace(0.0, wave.depth, 5)
print(" s (m)   u_m (m/s)   excursion (m)")
for si, ui, ai in zip(s, wave.u_m(s), wave.orbital_excursion(s), strict=True):
    print(f"{si:6.2f}   {ui:9.4f}   {ai:13.4f}")
print(f"\ndepth-averaged u_bar = {wave.u_bar:.4f} m/s")

The shape of this profile depends on $kh$. In the figure, a long wave (small $kh$)
moves the water almost uniformly, while a short wave (large $kh$) barely moves the water
near the bed. This matters for vegetation: **plants near the bed matter less under short
waves**, because the water there hardly moves.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for T, color in [(2.4, C1), (1.4, C2), (0.8, C3)]:
    w = sd.RegularWave(0.06, T, 0.4)
    s = np.linspace(0.0, w.depth, 100)
    ax.plot(w.u_m(s), s, color=color, lw=2, label=f"T = {T} s (kh = {w.kh:.2f})")
    ax.plot([w.u_bar] * 2, [0, w.depth], color=color, lw=1, ls=":")
ax.axhline(0.4, color="0.6", lw=1)
ax.text(0.005, 0.405, "still-water surface", fontsize=8, color="0.4", va="bottom")
ax.set_xlabel("velocity amplitude $u_m$ (m/s)")
ax.set_ylabel("height above bed $s$ (m)")
ax.set_ylim(0, 0.45)
ax.set_title("H = 6 cm, h = 0.4 m; dotted lines: depth averages $\\bar u$")
ax.legend(loc="lower right")
plt.show()

### 2.5 Energy and energy flux

A wave's energy per square metre of water surface is $E = \rho g H^2/8$ (theory Eq. 2.11),
and the rate at which that energy is carried forward, per metre of crest, is the
**energy flux** $E\,c_g$ (Eq. 2.12). Note that energy grows with the *square* of the
height: halving the wave height removes three quarters of its energy.

Both methods take a `Water` (default fresh, 20 °C).

In [ ]:
E = wave.energy(fresh)
F = wave.energy_flux(fresh)
print(f"energy      E     = {E:.3f} J/m²")
print(f"energy flux E c_g = {F:.3f} W/m")
print(f"check by hand: rho g H²/8 = {fresh.rho * sd.GRAVITY * wave.H**2 / 8:.3f} J/m²")

### 2.6 Dimensionless numbers

Wave and drag behaviour is described by ratios that have no units:

| Property | Meaning |
|---|---|
| `wave.steepness` | $H/L$, how steep the wave is |
| `wave.relative_height` | $H/h$, how tall the wave is compared with the depth |
| `wave.ursell` | Ursell number $\mathrm{Ur} = HL^2/h^3$; large values mean the wave is strongly nonlinear |

Two more, the **Reynolds number** $\mathrm{Re} = u\ell/\nu$ and the
**Keulegan–Carpenter number** $\mathrm{KC} = uT/\ell$, describe the flow around a plant
part. They need a velocity scale $u$ and a length scale $\ell$ (for example the stem
diameter), and the choice changes the value a lot. That is why SADIE never picks them
for you: the functions in `sadiepy.core.waves` take both explicitly.

In [ ]:
from sadiepy.core.waves import keulegan_carpenter, reynolds

d = 0.01  # a 1 cm stem
print(f"H/L = {wave.steepness:.4f}, H/h = {wave.relative_height:.3f}, Ur = {wave.ursell:.2f}")
print(f"Re_d = u_bar d / nu = {reynolds(wave.u_bar, d, fresh.nu):.0f}")
print(f"KC_d = u_bar T / d  = {keulegan_carpenter(wave.u_bar, wave.T, d):.1f}")

Roughly: KC compares how far the water moves in one stroke with the size of the
object (large KC: the flow sweeps far past the stem, like a steady current; small KC: it
just sloshes around it). Re compares inertia with viscosity (large Re: turbulent wake).

### 2.7 Is linear wave theory valid for this wave?

SADIE uses **linear** wave theory, which assumes the wave is small. `validity_issues()`
returns a list of every limit (theory §2.10) the wave violates; an empty list means all
clear:

- **depth-limited breaking**: $H/h > 0.78$;
- **steepness-limited breaking**: $H/L > 0.142\tanh(kh)$;
- **nonlinearity**: the wave is so nonlinear that second-order theory predicts a bump in
  the trough (Dean & Dalrymple 1991, Eq. 11.41).

In [ ]:
print("our wave:", wave.validity_issues())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    big = sd.RegularWave(H=0.16, T=2.0, depth=0.5)  # long, high wave
print("\nbig wave issues:")
for issue in big.validity_issues():
    print(" -", issue)
print("warnings emitted when it was created:", [w.category.__name__ for w in caught])

If you want a stricter, project-specific limit on the Ursell number, pass
`ursell_max`; waves above it are then flagged too:

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", sd.ValidityWarning)
    strict = sd.RegularWave(H=0.08, T=1.4, depth=0.4, ursell_max=5.0)
print(strict.validity_issues())

## 3. The vegetation

### 3.1 One layer of plant parts

SADIE describes vegetation as a **vertical stack of layers**. A layer is a group of
identical elements (stems, roots, leaves, ...) that occupy the same range of heights.
For each layer you give:

| Argument | Meaning | Units |
|---|---|---|
| `name` | a label, unique within the plant | – |
| `s_bottom`, `s_top` | bottom and top of the layer, above the bed | m |
| `n_per_m2` | number of elements per square metre of bed | 1/m² |
| `frontal_width` | width of one element as seen by the oncoming flow | m |
| `solid_area` (optional) | cross-section of one element seen from above; needed only for volumes | m² |
| `drag` (optional) | a drag law just for this layer (section 4.5) | – |

The **frontal width** is the key geometric quantity: drag is proportional to the area the
flow hits. For a cylinder it is just the diameter.

Here is a single layer of 1 cm stems, 40 cm tall, at 36 stems per m²:

In [ ]:
stems = sd.Layer(
    "stem",
    s_bottom=0.0,
    s_top=0.40,
    n_per_m2=36.0,
    frontal_width=0.01,
    solid_area=math.pi * 0.01**2 / 4,  # circle of diameter 1 cm
)
stems

A layer can tell you how much of it is underwater for a given depth. Only the
submerged part touches the moving water, so only that part takes energy from the wave.

In [ ]:
for h in (0.3, 0.4, 0.5):
    print(f"h = {h} m: submerged range {stems.submerged_range(h)}, "
          f"frontal area per stem {stems.frontal_area(h) * 1e4:.0f} cm²")

### 3.2 Stacking layers into a plant

A `Vegetation` is an ordered tuple of layers, plus optional metadata. Let's add a
dense layer of thin roots near the bed, like a simple mangrove:

- `reference_diameter` is the diameter used by drag laws based on "$d$" (for a mangrove,
  the trunk diameter at breast height). Set it if you plan to use such a law.
- `name`, `source` and `notes` are free text for your own records.

Layers may overlap in height (here the stem passes through the root zone).

In [ ]:
roots = sd.Layer("root", 0.0, 0.10, n_per_m2=1008.0, frontal_width=0.005,
                 solid_area=math.pi * 0.005**2 / 4)
plant = sd.Vegetation(
    layers=(roots, stems),
    name="roots + stems",
    source="illustrative values, not a published model",
    reference_diameter=0.01,
)
print(plant.name, "| layers:", [layer.name for layer in plant.layers])
print("height of the tallest layer:", plant.height, "m")

The most useful single picture of a plant is its **frontal area per unit volume**,
$a(s) = \sum_i N_i\, b_i(s)$: at each height, how much plant surface faces the flow in
one cubic metre of water. `frontal_area_profile(s)` returns it. Wherever $a(s)$ is large
*and* the water moves fast, energy is lost.

In [ ]:
s = np.linspace(0, 0.45, 451)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(plant.frontal_area_profile(s), s, color=C1, lw=2)
for h, ls in [(0.3, ":"), (0.5, "--")]:
    ax.axhline(h, color="0.5", ls=ls, lw=1)
    ax.text(1.0, h + 0.005, f"water surface if h = {h} m", fontsize=8, color="0.4")
ax.set_xlabel("frontal area per unit volume $a(s)$ (1/m)")
ax.set_ylabel("height above bed $s$ (m)")
ax.set_title("Roots + stems")
plt.show()

Dense roots make the bottom 10 cm about fifteen times "thicker" than the stem zone.
With $h = 0.3$ m the plant sticks out of the water (it is **emergent**); with $h = 0.5$ m
it is fully **submerged**.

### 3.3 Widths that change with height

Real plant parts are rarely uniform. `frontal_width` (and `solid_area`) may also be a
**function of height** $s$. A mangrove canopy, for example, can be represented as an
inverted triangle: zero width at its base, widening upwards. Here it sits between 0.23 m
and 0.40 m and reaches 0.145 m wide at the top.

The function receives a height in metres (above the bed) and returns a width in metres.
It only has to be valid inside the layer.

In [ ]:
def canopy_width(s):
    return 0.145 * (s - 0.23) / 0.17  # 0 at s = 0.23 m, 0.145 m at s = 0.40 m


canopy = sd.Layer("canopy", 0.23, 0.40, n_per_m2=36.0, frontal_width=canopy_width)
tree = sd.Vegetation(layers=(roots, stems, canopy), name="roots + stems + canopy",
                     reference_diameter=0.01)

print("constant width?", canopy.has_constant_width)
print("width at s = 0.30 m:", round(canopy.width_at(0.30), 4), "m")

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(tree.frontal_area_profile(s), s, color=C3, lw=2, label="with canopy")
ax.plot(plant.frontal_area_profile(s), s, color=C1, lw=1.5, ls="--", label="without")
ax.set_xlabel("frontal area per unit volume $a(s)$ (1/m)")
ax.set_ylabel("height above bed $s$ (m)")
ax.legend()
plt.show()

Behind the scenes, constant-width layers use an exact formula and variable-width
layers are integrated numerically (theory §3.4). The two agree to better than
$10^{-8}$, so you never have to choose.

### 3.4 Volumes and characteristic lengths

Some published drag laws use length scales built from the plant's volume and frontal
area (theory §5.1–5.2). These need `solid_area` on every layer and a
`reference_diameter`. All of them depend on the water depth, so each takes `h`:

| Method | Meaning |
|---|---|
| `frontal_area(h)` | submerged frontal area per m² of bed, $A_f$ |
| `submerged_volume(h)` | submerged solid volume per m² of bed, $V_s$ |
| `svf(h)` | solid volume fraction, $V_s/h$ |
| `l_e(h)` | effective vegetation length $V_s/A_f$ |
| `l_ew(h)` | effective water length $(h - V_s)/A_f$ |
| `hydraulic_radius(h)` | vegetation hydraulic radius $r_v$ (Cheng & Nguyen 2011) |

In [ ]:
h = 0.4
print(f"at h = {h} m:")
print(f"  A_f  = {plant.frontal_area(h):.4f} m²/m²")
print(f"  V_s  = {plant.submerged_volume(h) * 1e3:.3f} litres/m²")
print(f"  SVF  = {plant.svf(h):.4f}")
print(f"  l_e  = {plant.l_e(h) * 1e3:.2f} mm")
print(f"  l_ew = {plant.l_ew(h):.3f} m")
print(f"  r_v  = {plant.hydraulic_radius(h):.3f} m")

(`tree` has no `solid_area` on its canopy, so asking it for a volume raises an
error. Section 3.5 shows a preset that has everything filled in.)

### 3.5 Presets

Writing layers by hand is fine for simple cases. For published models, SADIE has
**presets**: functions that return a ready-made `Vegetation` with every number traced to
its source.

- `sd.presets.rigid_cylinders(d, n_per_m2, height)`: a uniform array of rods, the
  standard laboratory test case.
- `sd.presets.zhang2023(config, scale)`: the 1:10 artificial mangroves of Zhang et al.
  (2023), *Coastal Engineering* 186, 104403. `config` is `"M1"` (roots + stem), `"M2"`
  (+ sparse canopy) or `"M3"` (+ dense canopy); `scale` is `"model"` (as built) or
  `"prototype"` (10 times larger).

In [ ]:
rods = sd.presets.rigid_cylinders(d=0.01, n_per_m2=400.0, height=0.3)
print(rods.name)

m3 = sd.presets.zhang2023("M3")
print("\n" + m3.name)
for layer in m3.layers:
    width = "varies with s" if not layer.has_constant_width else f"{layer.frontal_width} m"
    print(f"  {layer.name:7s} {layer.s_bottom:.2f}-{layer.s_top:.2f} m, "
          f"{layer.n_per_m2:6.0f} /m², width {width}")
print(f"  SVF at h = 0.4 m: {m3.svf(0.4):.4f}")

The preset's docstring (`help(sd.presets.zhang2023)`) lists every parameter with
the table, figure and page of the paper it comes from.

### 3.6 Simplified (averaged) geometries

Published analyses often simplify the plant before computing damping. SADIE offers the
two most common simplifications as transformations, so you can compare them with the
full description using the same solver (theory §5.3):

- `veg.layer_averaged(h)`: each layer keeps its height range, but a varying width is
  replaced by its average over the submerged part (Zhang et al. 2023, Eqs. 19–20).
- `veg.depth_averaged(h)`: the whole plant becomes **one** uniform layer whose frontal
  area per unit volume is $A_f(h)/h$ (Zhang et al. 2023, Eq. 4). This is the geometry
  under which most published drag laws were fitted.

Both depend on the water depth, so the result is tied to that depth, and the solver
refuses to use it with a wave in a different depth.

In [ ]:
h = 0.4
versions = {
    "full": m3,
    "layer-averaged": m3.layer_averaged(h),
    "depth-averaged": m3.depth_averaged(h),
}
s = np.linspace(0, h, 401)
fig, ax = plt.subplots(figsize=(5, 4))
for (label, veg), color in zip(versions.items(), (C1, C2, C3), strict=True):
    ax.plot(veg.frontal_area_profile(s), s, color=color, lw=2, label=label)
ax.set_xlabel("frontal area per unit volume $a(s)$ (1/m)")
ax.set_ylabel("height above bed $s$ (m)")
ax.set_title("Zhang et al. (2023) M3 at h = 0.4 m, three descriptions")
ax.legend()
plt.show()

for label, veg in versions.items():
    print(f"{label:15s} total frontal area A_f = {veg.frontal_area(h):.4f} m²/m², "
          f"derived for h = {veg.derived_depth}")

All three have the same total frontal area; they differ in *where* it is. Because
the water moves faster near the surface, moving area upwards increases damping, so the
choice of description changes the answer even with the same drag coefficient.

## 4. The drag coefficient

### 4.1 What $C_D$ is

The force of the flow on a plant part is

$$F = \tfrac12\,\rho\,C_D\,(\text{frontal area})\,u\,|u|.$$

Everything in this formula is known from the water, the wave and the plant, except the
**drag coefficient** $C_D$. It depends on the shape of the element, on how elements
shelter each other, and on the flow (Re, KC). It cannot be derived from first
principles; it comes from experiments. Typical values for vegetation are of order 1, but
values from 0.5 to 10 are reported.

SADIE accepts $C_D$ in three forms, explained below: a plain number, a named published
law, or a law you write yourself.

### 4.2 A constant

The simplest choice: pass a number, e.g. `drag=1.0`, to the solver. Internally this
becomes a `DragClosure` made by `sd.drag.constant`:

In [ ]:
cd = sd.drag.constant(1.2)
print(cd.name, "| variable:", cd.variable, "| C_D =", cd())

### 4.3 Published drag laws (the registry)

A **closure** is a formula for $C_D$ as a function of one dimensionless number (Re, KC or
Ur). SADIE includes ten closures for mangrove-like vegetation, transcribed from
Zhang et al. (2023), Table 3. `sd.drag.names()` lists them; `sd.drag.get(name)` returns
one.

In [ ]:
for name in sd.drag.names():
    c = sd.drag.get(name)
    print(f"{name:15s} C_D({c.variable}), length {str(c.length_scale):5s} "
          f"valid {c.valid_range[0]:>8g} to {c.valid_range[1]:<8g} source scale 1:{c.scale:g}")

Each closure carries everything needed to use it correctly. Let's look at one in
detail:

In [ ]:
c = sd.drag.get("zhang2023_kc")
print("name          :", c.name)
print("variable      :", c.variable, "(the number C_D depends on)")
print("velocity scale:", c.velocity_scale, "(depth-averaged velocity)")
print("length scale  :", c.length_scale, "(the vegetation's reference diameter)")
print("valid range   :", c.valid_range, "(range of KC it was fitted to)")
print(f"source scale  : 1:{c.scale:g}")
print("components    :", c.components)
print("citation      :", c.citation)

A closure is callable: give it the value of its variable and it returns $C_D$.
`in_range` tells you whether that value is inside the fitted range. Outside the range
the formula still evaluates, but the result is an extrapolation; when the solver does
this it records it and warns (section 6.3).

In [ ]:
for kc in (10.0, 30.0, 100.0):
    print(f"KC = {kc:5.0f}: C_D = {c(kc):.3f}, inside fitted range? {c.in_range(kc)}")

kc = np.linspace(2, 100, 300)
lo, hi = c.valid_range
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.axvspan(lo, hi, color="#cde2fb", label="fitted range")
ax.plot(kc, [c(k) for k in kc], color=C1, lw=2, label=c.name)
ax.set_xlabel("KC")
ax.set_ylabel("$C_D$")
ax.set_ylim(0, 12)
ax.legend()
plt.show()

### 4.4 Your own drag law

`sd.drag.custom` wraps any Python function as a closure. You must say which variable it
uses and, for Re and KC, which length scale (a number in metres, or one of `"d"`,
`"l_e"`, `"l_ew"`, `"r_v"`). Giving a `valid_range` and a `citation` is optional but
recommended: SADIE then warns you when you leave the range and records where the law
came from.

The law below is **made up for this tutorial**; it is not from any paper.

In [ ]:
my_law = sd.drag.custom(
    "tutorial_kc",
    lambda kc: 1.0 + 10.0 / kc,
    variable="KC",
    length_scale="d",
    valid_range=(5.0, 80.0),
    citation="made-up example for the tutorial",
)
print(my_law.name, "| C_D at KC = 20:", my_law(20.0))

### 4.5 One $C_D$ for the whole plant, or one per layer?

There are two ways to attach drag laws, and SADIE records which one a result used:

- **Bulk** (the usual case): pass `drag=...` to `sd.propagate`. The same closure is used
  for every layer. This matches how published closures were fitted: one $C_D$ for the
  whole plant.
- **Per-layer**: give each `Layer` its own `drag=...` and pass nothing to `propagate`.
  Useful when you have separate measurements for, say, stems and leaves.

You cannot mix the two (section 8 shows the error). We use both in section 5.6.

## 5. Propagating a wave through the patch

### 5.1 The simplest call

`sd.propagate(wave, vegetation, patch_length, ...)` is the main function. The wave is the
incident wave at the leading edge ($x = 0$); `patch_length` is the length of the patch in
the direction the wave travels.

We'll send our 8 cm wave through 2 m of the roots + stems plant, with $C_D = 1$.

In [ ]:
res = sd.propagate(wave, plant, patch_length=2.0, water=fresh, drag=1.0)
print("positions x  :", res.x, "m")
print("wave heights :", res.H.round(5), "m")
print(f"damping factor beta = {res.beta:.5f} 1/m")
print(f"transmitted fraction H(2 m)/H0 = {res.H[-1] / wave.H:.3f}")

With a constant $C_D$ the wave height follows a simple curve (theory Eq. 3.7):

$$H(x) = \frac{H_0}{1 + \beta x},$$

where $\beta$ (units 1/m) is the **damping factor**. One number summarises how strongly
the patch damps this wave: a larger $\beta$ means faster decay. After a distance
$x = 1/\beta$ the wave is half its original height.

### 5.2 Everything in the result

`propagate` returns a `PropagationResult`. Besides `H` and `beta`, it reports how the
answer was obtained, so a result can always be explained later:

In [ ]:
for field in dataclasses.fields(res):
    value = getattr(res, field.name)
    if isinstance(value, np.ndarray):
        value = f"array of {value.size} values"
    print(f"{field.name:23s} {value}")

In words:

- `x`, `H`: positions and wave heights; `beta`: damping factor.
- `beta_fit_rms`: when $\beta$ is fitted rather than exact (section 6), the misfit of the
  curve. It is 0 here because the solution is exact.
- `layer_fractions`: the share of the energy lost in each layer.
- `cumulative_dissipation`, `energy_flux`: the energy budget (section 5.4).
- `method`: `"analytic"` (exact formula) or `"ode"` (numerical integration).
- `drag_mode`, `drag_used`, `drag_coefficients`: which drag law each layer used, and the
  range of $C_D$ values it produced.
- `cd_evaluation`, `scale`, `extrapolated`, `dimensionless`: how wave-dependent drag laws
  were evaluated (section 6).
- `warnings`: every validity or extrapolation message raised during the calculation.

### 5.3 Asking for more positions

By default you get $x = 0$ and the end of the patch. Pass `x_out` for any positions inside
the patch:

In [ ]:
x = np.linspace(0.0, 2.0, 81)
res = sd.propagate(wave, plant, 2.0, water=fresh, drag=1.0, x_out=x)
res_tree = sd.propagate(wave, tree, 2.0, water=fresh, drag=1.0, x_out=x)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(x, res.H * 100, color=C1, lw=2, label=f"roots + stems (β = {res.beta:.3f} 1/m)")
ax.plot(x, res_tree.H * 100, color=C3, lw=2, label=f"+ canopy (β = {res_tree.beta:.3f} 1/m)")
ax.set_xlabel("distance into the patch $x$ (m)")
ax.set_ylabel("wave height $H$ (cm)")
ax.set_ylim(0, 8.5)
ax.legend()
plt.show()

### 5.4 Checking the answer by hand

It is worth seeing that there is no magic in that number. For a single uniform layer of
cylinders standing on the bed, $\beta$ has a closed form (Zhang et al. 2023, Eq. 4;
theory Eq. 3.8):

$$\beta = \frac{4}{9\pi}\,C_D\,D\,N\,H_0\,k\,
\frac{\sinh^3(kh_v) + 3\sinh(kh_v)}{(\sinh 2kh + 2kh)\,\sinh kh},$$

with $D$ the diameter, $N$ the number per m² and $h_v$ the submerged height. Let's
compute it for the stems alone and compare with the solver:

In [ ]:
C_D, D, N, hv = 1.0, 0.01, 36.0, min(0.40, wave.depth)
k, kh = wave.k, wave.kh
S = math.sinh(k * hv) ** 3 + 3 * math.sinh(k * hv)
beta_hand = 4 / (9 * math.pi) * C_D * D * N * wave.H * k * S / (
    (math.sinh(2 * kh) + 2 * kh) * math.sinh(kh))

stems_only = sd.Vegetation(layers=(stems,))
beta_sadie = sd.propagate(wave, stems_only, 2.0, water=fresh, drag=1.0).beta
print(f"by hand: {beta_hand:.10f} 1/m")
print(f"SADIE  : {beta_sadie:.10f} 1/m")
print(f"relative difference: {abs(beta_sadie / beta_hand - 1):.1e}")

(Water density cancels out of $\beta$, which is why the hand formula has no $\rho$.)

### 5.5 The energy budget

SADIE tracks energy explicitly. `energy_flux` is $E c_g$ at each position and
`cumulative_dissipation` is the energy the plants have removed between $0$ and $x$
(both in W per metre of crest). Energy is conserved, so what the wave loses must equal
what the plants dissipate:

In [ ]:
lost_by_wave = res.energy_flux[0] - res.energy_flux[-1]
taken_by_plants = res.cumulative_dissipation[-1]
print(f"energy flux in  : {res.energy_flux[0]:.4f} W/m")
print(f"energy flux out : {res.energy_flux[-1]:.4f} W/m")
print(f"lost by wave    : {lost_by_wave:.6f} W/m")
print(f"taken by plants : {taken_by_plants:.6f} W/m")
print(f"budget error    : {abs(lost_by_wave - taken_by_plants) / lost_by_wave:.1e}")

### 5.6 Which part of the plant does the work?

`layer_fractions` splits the dissipated energy between the layers. With a constant
$C_D$, the damping factors of separate layers simply **add up** (theory §3.7), so you can
also run each layer on its own and sum the results:

In [ ]:
print("shares:", {name: round(f, 3) for name, f in res_tree.layer_fractions.items()})

beta_parts = {}
for layer in tree.layers:
    alone = sd.Vegetation(layers=(layer,))
    beta_parts[layer.name] = sd.propagate(wave, alone, 2.0, water=fresh, drag=1.0).beta
print("beta of each layer alone:", {k: round(v, 4) for k, v in beta_parts.items()})
print(f"sum = {sum(beta_parts.values()):.6f} 1/m;  whole tree = {res_tree.beta:.6f} 1/m")

fig, ax = plt.subplots(figsize=(5.5, 1.6))
left = 0.0
for name, frac in res_tree.layer_fractions.items():
    ax.barh(0, frac, left=left, color=PART[name], edgecolor="white", linewidth=2)
    ax.text(left + frac / 2, 0, f"{name}\n{frac:.0%}", ha="center", va="center",
            color="black" if name == "stem" else "white", fontsize=9)
    left += frac
ax.set_xlim(0, 1)
ax.axis("off")
ax.set_title("Share of the energy lost, by layer (C_D = 1)", fontsize=10)
plt.show()

Now the **per-layer** mode: give each layer its own drag law. Here (as an
illustration only) the roots get $C_D = 2$, the stem $1$ and the canopy $0.5$. Note that
`propagate` gets no `drag` argument this time.

In [ ]:
tree_per_layer = sd.Vegetation(layers=(
    dataclasses.replace(roots, drag=sd.drag.constant(2.0)),
    dataclasses.replace(stems, drag=sd.drag.constant(1.0)),
    dataclasses.replace(canopy, drag=sd.drag.constant(0.5)),
))
r = sd.propagate(wave, tree_per_layer, 2.0, water=fresh)
print("drag mode:", r.drag_mode)
print("C_D used :", {k: v[0] for k, v in r.drag_coefficients.items()})
print(f"beta = {r.beta:.4f} 1/m")

## 6. When the drag coefficient depends on the wave

### 6.1 Why this is different

Published closures give $C_D$ as a function of KC, Re or Ur, which depend on the wave
height. As the wave shrinks along the patch, these numbers change, so $C_D$ changes too.
Then the simple curve $H_0/(1 + \beta x)$ no longer holds exactly, and SADIE integrates
the energy balance step by step along the patch instead (theory Eq. 3.9;
`method = "ode"`). It still reports a $\beta$: the best fit of the simple curve to the
computed heights, with the misfit in `beta_fit_rms`.

Two options control how a closure is used:

- `cd_evaluation`:
  - `"local"` (default): $C_D$ is re-evaluated at every point from the local wave height.
    This is the physically natural choice.
  - `"incident"`: $C_D$ is evaluated once, from the incident wave, and kept constant along
    the patch. This is how published bulk closures were fitted, so it is the consistent
    way to apply them.
- `scale`: the geometric scale $1:\lambda$ of *your* case (e.g. `10` for a 1:10 model,
  `1` for full size). It is used only by Re-based closures (section 7.3).

Let's use our made-up `my_law` on the roots + stems + canopy tree both ways, over a
10 m patch so that the wave shrinks enough for the difference to show:

In [ ]:
x = np.linspace(0.0, 10.0, 201)
local = sd.propagate(wave, tree, 10.0, water=fresh, drag=my_law, x_out=x)
incident = sd.propagate(wave, tree, 10.0, water=fresh, drag=my_law, x_out=x,
                        cd_evaluation="incident")

for label, r in [("local", local), ("incident", incident)]:
    lo, hi = r.drag_coefficients["stem"]
    print(f"{label:9s} method={r.method:8s} beta={r.beta:.4f} 1/m "
          f"(fit rms {r.beta_fit_rms:.1e}), C_D from {lo:.3f} to {hi:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(x, local.H / wave.H, color=C1, lw=2, label="local: C_D follows H(x)")
ax.plot(x, incident.H / wave.H, color=C2, lw=2, ls="--", label="incident: C_D fixed at H0")
ax.set_xlabel("distance into the patch $x$ (m)")
ax.set_ylabel("$H/H_0$")
ax.legend()
plt.show()

With `"local"`, KC falls as the wave shrinks, `my_law` gives a larger $C_D$ at smaller
KC, so by the end of the patch $C_D$ has risen from about 1.4 to 1.9 and the wave decays
faster than with the incident value. The decay is then no longer exactly
$H_0/(1+\beta x)$, which shows up as a non-zero `beta_fit_rms`.

### 6.2 What the closure saw

`result.dimensionless` records the dimensionless numbers the closures were evaluated
at, with the velocity and length scales used. This is what lets you check a result
against the closure's fitted range later.

In [ ]:
for key, value in local.dimensionless.items():
    print(f"{key}: {value}")

### 6.3 Using a published closure, and extrapolation

Published closures come with a fitted range. When a calculation evaluates a closure
outside that range, SADIE still computes the result but sets `result.extrapolated = True`,
adds a message to `result.warnings` and emits an `sd.ExtrapolationWarning`.

The closures of Zhang et al. (2023) were fitted with the depth-averaged geometry
(section 3.6), evaluated at the incident wave, at the 1:10 model scale. Applying
`zhang2023_kc` to their M3 model *the same way* stays inside its fitted range. (The
depth-averaged plant has a single layer, named `"depth-averaged"`.)

In [ ]:
h = 0.4
wave_04 = sd.RegularWave(0.08, 1.4, h)
consistent = sd.propagate(
    wave_04, m3.depth_averaged(h), 2.0, drag="zhang2023_kc",
    cd_evaluation="incident", scale=10.0,
)
print("layers:", list(consistent.drag_coefficients))
cd_value = consistent.drag_coefficients["depth-averaged"][0]
print(f"beta = {consistent.beta:.3f} 1/m, C_D = {cd_value:.2f}")
print("KC range:", consistent.dimensionless["KC[zhang2023_kc]"]["range"])
print("extrapolated?", consistent.extrapolated)

Here is a case that leaves the range: a small wave in deep water gives a KC below the
fitted minimum. The warning is caught and printed so you can see it:

In [ ]:
small = sd.RegularWave(0.02, 1.0, 0.5)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    r = sd.propagate(small, m3.depth_averaged(0.5), 2.0, drag="zhang2023_kc",
                     cd_evaluation="incident", scale=10.0)
print("extrapolated?", r.extrapolated)
for message in r.warnings:
    print(" -", message)
print("warnings emitted:", sorted({w.category.__name__ for w in caught}))

Warnings are standard Python warnings, so you control them with the `warnings`
module. In a script where an extrapolated result should stop the run, turn them into
errors:

```python
warnings.simplefilter("error", sd.ExtrapolationWarning)  # only closure extrapolation
warnings.simplefilter("error", sd.ValidityWarning)       # every validity problem
```

Whatever you do with the warnings, `result.extrapolated` and `result.warnings` keep the
record.

## 7. Scaling between the lab and the field

### 7.1 Froude scaling

Flume experiments are usually built at a reduced scale $1:\lambda$ (e.g. 1:10). Waves are
scaled so that the ratio of inertia to gravity (the Froude number) is the same in the
model and in reality. Each quantity then scales with a fixed power of $\lambda$
(theory §4). `sadiepy.scaling.froude` provides these factors and conversions:

In [ ]:
from sadiepy.scaling import froude

lam = 10.0
print("prototype / model, for a 1:10 model:")
for quantity in froude.EXPONENTS:
    print(f"  {quantity:16s} x {froude.scale_factor(quantity, lam):g}")

So a 1.4 s wave in the flume is a 4.4 s wave at full size, KC and Ur are unchanged,
and the Reynolds number is about 32 times larger. `to_prototype` and `to_model` convert
single values; `wave_to_prototype` and `vegetation_to_prototype` convert whole objects.

In [ ]:
print("1.4 s in the model =", round(froude.to_prototype(1.4, "time", lam), 3), "s at full size")

wave_p = froude.wave_to_prototype(wave_04, lam)
m3_p = sd.presets.zhang2023("M3", scale="prototype")
print(f"prototype wave: H = {wave_p.H:.2f} m, T = {wave_p.T:.3f} s, h = {wave_p.depth:.1f} m")
print(f"KC is unchanged: model KC_d = {keulegan_carpenter(wave_04.u_bar, wave_04.T, 0.01):.3f}, "
      f"prototype KC_d = {keulegan_carpenter(wave_p.u_bar, wave_p.T, 0.1):.3f}")

### 7.2 Damping at full size

With a constant $C_D$, $\beta$ has units of 1/m and scales as $1/\lambda$: the full-size
forest damps the wave over a patch ten times longer. SADIE reproduces this exactly:

In [ ]:
b_model = sd.propagate(wave_04, m3, 2.0, drag=1.0).beta
b_proto = sd.propagate(wave_p, m3_p, 20.0, drag=1.0).beta
print(f"model     beta = {b_model:.5f} 1/m over a 2 m patch")
print(f"prototype beta = {b_proto:.5f} 1/m over a 20 m patch")
print(f"ratio = {b_model / b_proto:.6f}  (expected lambda = {lam:g})")

### 7.3 Reynolds-number closures at another scale

KC and Ur are the same at every scale, but Re is not. Take `maza2017_re`, fitted in a 1:12
flume. Applied to a full-size forest, it would see a Reynolds number $12^{3/2} \approx 42$
times larger than the Froude-similar flow it was fitted to, far outside its range. Following Zhang et al. (2023, §4.4) and
Kelty et al. (2022), SADIE can convert Re to the closure's source scale first
(theory Eq. 5.6). It does so **only when you pass `scale`**, and it records the factor it
used:

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", sd.ExtrapolationWarning)
    converted = sd.propagate(wave_p, m3_p.depth_averaged(wave_p.depth), 20.0,
                             drag="maza2017_re", cd_evaluation="incident", scale=1.0)
    raw = sd.propagate(wave_p, m3_p.depth_averaged(wave_p.depth), 20.0,
                       drag="maza2017_re", cd_evaluation="incident")
for label, r in [("scale=1 (converted)", converted), ("no scale (raw)", raw)]:
    info = r.dimensionless["Re[maza2017_re]"]
    cd_value = r.drag_coefficients["depth-averaged"][0]
    print(f"{label:20s} Re seen by closure = {info['range'][0]:9.0f}, "
          f"factor = {info['re_scale_factor']:.4f}, C_D = {cd_value:.3f}")
print("fitted Re range of maza2017_re:", sd.drag.get("maza2017_re").valid_range)

Neither option is "right" in general: converting assumes $C_D$ behaves as it did in the
Froude-scaled source experiment; not converting assumes the closure holds at the real Re.
SADIE makes you choose and records the choice.

## 8. Errors and what they mean

SADIE raises an error (instead of a warning) when an input makes no sense. Each message
says what to fix. The most common ones:

In [ ]:
def show_error(label, func):
    try:
        func()
    except (ValueError, KeyError) as err:
        print(f"{label}\n  {type(err).__name__}: {err}\n")


show_error("A layer with its top below its bottom:",
           lambda: sd.Layer("bad", 0.3, 0.1, 10.0, 0.01))
show_error("No drag law anywhere:",
           lambda: sd.propagate(wave, plant, 2.0))
show_error("Bulk drag and per-layer drag at the same time:",
           lambda: sd.propagate(wave, tree_per_layer, 2.0, drag=1.0))
show_error("Output position outside the patch:",
           lambda: sd.propagate(wave, plant, 2.0, drag=1.0, x_out=[3.0]))
show_error("Averaged geometry for a different depth:",
           lambda: sd.propagate(wave, m3.depth_averaged(0.5), 2.0, drag=1.0))
show_error("Forcing the exact formula with a wave-dependent C_D:",
           lambda: sd.propagate(wave, plant, 2.0, drag=my_law, method="analytic"))
show_error("A misspelled closure name:",
           lambda: sd.drag.get("zhang_kc"))
show_error("Negative salinity:",
           lambda: sd.Water(salinity=-1.0))

## 9. Putting it together: a small study

A typical use of SADIE is to run many wave conditions through one or more plants and
compare. Here we take the wave conditions Zhang et al. (2023) used at $h = 0.4$ m
(`TEST_MATRIX`, from their Table 2) and compute, for each of their three models, the
damping factor and the fraction of wave height that gets through a 2 m patch.

We write a small helper so each calculation is one line, and use the closure the way it
was fitted (depth-averaged geometry, incident evaluation, model scale 1:10).

In [ ]:
from sadiepy.presets.zhang2023 import TEST_MATRIX

MODELS = {name: sd.presets.zhang2023(name) for name in ("M1", "M2", "M3")}


def damping(model, H, T, h, patch=2.0):
    # Damping factor (1/m) and transmitted fraction H_out/H0 for one test.
    # Warnings are recorded in r.warnings, so silence the printed copies here.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", sd.ValidityWarning)
        w = sd.RegularWave(H, T, h)
        r = sd.propagate(w, MODELS[model].depth_averaged(h), patch, drag="zhang2023_kc",
                         cd_evaluation="incident", scale=10.0)
    return r.beta, r.H[-1] / H, r.warnings


tests = [t for t in TEST_MATRIX if t.h == 0.4]
print(f"{len(tests)} wave conditions at h = 0.4 m\n")
print("  H (m)  T (s)    beta M1  M2     M3     H_out/H0 M1  M2    M3")
rows = []
for t in tests:
    out = {m: damping(m, t.H, t.T, t.h) for m in MODELS}
    rows.append((t, out))
    flagged = sorted({msg.split(",")[0] for o in out.values() for msg in o[2]})
    flag = "  * " + "; ".join(flagged) if flagged else ""
    print(f"  {t.H:.2f}   {t.T:.1f}    "
          + "  ".join(f"{out[m][0]:.3f}" for m in MODELS) + "       "
          + "  ".join(f"{out[m][1]:.2f}" for m in MODELS) + flag)
print("\n* = the result carries a warning (shown truncated)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
colors, markers = {"M1": C1, "M2": C2, "M3": C3}, {"M1": "o", "M2": "s", "M3": "^"}
for m in MODELS:
    for H, ls in [(0.06, ":"), (0.08, "-"), (0.10, "--")]:
        pts = sorted((t.T, out[m][0]) for t, out in rows if t.H == H)
        T, b = zip(*pts, strict=True)
        ax.plot(T, b, color=colors[m], ls=ls, marker=markers[m], ms=6, lw=1.5)
for m in MODELS:
    ax.plot([], [], color=colors[m], marker=markers[m], label=m)
for H, ls in [(0.06, ":"), (0.08, "-"), (0.10, "--")]:
    ax.plot([], [], color="0.4", ls=ls, label=f"H = {H} m")
ax.set_xlabel("wave period T (s)")
ax.set_ylabel("damping factor β (1/m)")
ax.set_title("Predicted damping, h = 0.4 m, closure zhang2023_kc")
ax.legend(ncol=2, fontsize=8)
plt.show()

One condition is flagged: the longest, highest wave (H = 0.10 m, T = 2.0 s) is beyond the
nonlinearity limit of section 2.7, so linear wave theory, and therefore this prediction,
is less reliable there. Checking `r.warnings` in a loop like this is good practice.

Reading the figure: the canopy (M2, M3) adds a lot of damping because it sits high in
the water column, where the water moves fastest (section 2.4), and denser canopy (M3)
damps more than sparse canopy (M2).

## 10. Cheat sheet

```python
import sadiepy as sd

water = sd.Water(temperature=20.0, salinity=0.0)          # °C, g/kg
wave = sd.RegularWave(H=0.08, T=1.4, depth=0.4)           # m, s, m
wave.k, wave.L, wave.cg, wave.u_m(s), wave.u_bar          # linear wave theory
wave.validity_issues()                                    # [] if linear theory is OK

layer = sd.Layer("stem", s_bottom=0.0, s_top=0.4, n_per_m2=36,
                 frontal_width=0.01,                      # float or function of s
                 solid_area=3.14e-4 * 0.25)               # optional, for volumes
veg = sd.Vegetation(layers=(layer,), reference_diameter=0.01)
veg = sd.presets.zhang2023("M3")                          # or a preset
veg.frontal_area_profile(s), veg.svf(h), veg.l_e(h), veg.hydraulic_radius(h)
veg.depth_averaged(h), veg.layer_averaged(h)              # simplified geometries

sd.drag.names(); sd.drag.get("zhang2023_kc")              # published closures
sd.drag.custom("mine", f, variable="KC", length_scale="d", valid_range=(5, 80))

res = sd.propagate(wave, veg, patch_length=2.0, water=water,
                   drag=1.0,                 # number, closure, or registered name
                   x_out=[0, 0.5, 1, 1.5, 2],
                   cd_evaluation="local",    # or "incident"
                   scale=None)               # 1:lambda of this case, for Re closures
res.H, res.beta, res.layer_fractions, res.energy_flux, res.cumulative_dissipation
res.method, res.drag_used, res.drag_coefficients, res.dimensionless
res.extrapolated, res.warnings

from sadiepy.scaling import froude
froude.scale_factor("time", 10); froude.wave_to_prototype(wave, 10)
```

**Where next**

- The [tour](02_tour.ipynb) shows the same tools on larger examples with more figures.
- The [theory](../theory.md) page derives every equation used here.
- The API reference documents every function and argument.